In [5]:
# -*- coding: utf-8 -*-
# =====================================================================================
#  0번 셀 — 라이브러리 설치
#  새 Colab T4 런타임에서 가장 먼저 한 번 실행합니다.
#  torch는 Colab 기본 설치본을 그대로 사용하며 재설치하지 않습니다(재설치 금지 규정).
# =====================================================================================
import subprocess
import sys


def _install_baseline_packages():
    packages = [
        "fastapi",
        "uvicorn",
        "langgraph",
        "pydantic>=2",
        "sentence-transformers",
        "faiss-cpu",
        "transformers",
        "accelerate",
        "bitsandbytes",
        "pypdf",
        "python-docx",
        "beautifulsoup4",
        "requests",
    ]
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *packages],
        check=True,
    )


_install_baseline_packages()
print("[0번 셀 완료] 라이브러리 설치 완료 — 다음 셀(결과기 로직)을 실행하세요.")


[0번 셀 완료] 라이브러리 설치 완료 — 다음 셀(결과기 로직)을 실행하세요.


In [6]:
# -*- coding: utf-8 -*-
"""
카카오 약관 RAG 결과기 — Baseline 구현본 (LangGraph)

스켈레톤의 모든 [STUB] 지점을 실제 로직으로 교체했다.
함수 시그니처와 스키마는 스켈레톤 원본을 그대로 유지하며, 각 함수가 필요로 하는
import는 모두 함수 본문 안에 둔다.

Baseline 구성
  로딩 requests + BeautifulSoup / 파싱 조 번호 정규식(순번 검증)
  청킹 조(article) 단위 / 임베딩 bge-m3 / 저장 FAISS IndexFlatIP
  검색 cosine top-k(k=4) / 생성 Qwen2.5-7B-Instruct 4bit
"""

from __future__ import annotations

import json
import threading
import time
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, Tuple

from langgraph.graph import END, StateGraph
from pydantic import BaseModel, ConfigDict, Field, field_validator, model_validator

# =====================================================================================
# 0. 스텁 헬퍼
# =====================================================================================

STUB_LOG: List[str] = []


def stub(stage: str, detail: str = "") -> None:
    """스텁 실행을 기록하고 출력한다. 실제 구현으로 교체할 지점 표시."""
    line = f"[STUB] {stage}" + (f" — {detail}" if detail else "")
    STUB_LOG.append(line)
    print(line)


# =====================================================================================
# 1. 고정 상수
# =====================================================================================

DocName = Literal[
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
]

OFFICIAL_DOCUMENT_NAMES: Tuple[str, ...] = (
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
)

REQUIRED_GENERATION_MODEL_FAMILY = "Qwen2.5-Instruct"


def normalize_doc_name(value: Any) -> str:
    """공통 러너 _sp_norm_doc과 동일 규칙 — NFC 정규화 + 공백 제거."""
    import re
    import unicodedata

    return re.sub(r"\s+", "", unicodedata.normalize("NFC", str(value)))


ALLOWED_DOCS_NORM = {normalize_doc_name(d) for d in OFFICIAL_DOCUMENT_NAMES}


# =====================================================================================
# 2. 스키마 — 설정
# =====================================================================================

class DocumentSource(BaseModel):
    """약관 1종의 원문 취득 경로. local_path가 있으면 로컬 파일을 우선 사용한다."""

    doc_name: DocName
    urls: List[str] = Field(default_factory=list)
    local_path: Optional[str] = None
    effective_date: str
    note: str = ""

    @model_validator(mode="after")
    def _require_source(self) -> "DocumentSource":
        if not self.urls and not self.local_path:
            raise ValueError(f"[{self.doc_name}] urls 또는 local_path 중 하나는 필요합니다.")
        return self


class IndexConfig(BaseModel):
    sources: List[DocumentSource]
    embedding_model_name: str = "BAAI/bge-m3"
    embedding_batch_size: int = 8
    max_chunk_chars: int = 1800
    request_timeout_s: float = 20.0


class RetrievalConfig(BaseModel):
    top_k: int = 4  # 채점 스키마 retrieved 상한과 일치
    query_prefix: str = ""


class GenerationConfig(BaseModel):
    model_name: str = "Qwen/Qwen2.5-7B-Instruct"
    load_in_4bit: bool = True
    max_new_tokens: int = 512
    temperature: float = 0.0
    max_context_chars: int = 6000


class PipelineConfig(BaseModel):
    index: IndexConfig
    retrieval: RetrievalConfig = RetrievalConfig()
    generation: GenerationConfig = GenerationConfig()


# =====================================================================================
# 3. 스키마 — 인덱싱
# =====================================================================================

class RawDocument(BaseModel):
    """취득한 약관 원문 1건 (평문 텍스트)."""

    doc_name: DocName
    text: str
    source_url: str
    fetched_at: str
    char_len: int


class Article(BaseModel):
    """조(條) 단위로 분해된 약관 조항."""

    doc_name: DocName
    article_number: int
    article_title: str = ""
    body: str

    @property
    def citation(self) -> str:
        """골드셋 gold_articles[].citation 과 같은 표기."""
        head = f"{self.doc_name} 제{self.article_number}조"
        return f"{head}({self.article_title})" if self.article_title else head


class Chunk(BaseModel):
    """벡터 저장소 최소 단위. 메타데이터가 채점 스키마와 직접 대응한다."""

    chunk_id: str
    doc_name: DocName
    article_number: int
    article_title: str = ""
    text: str


class EmbeddingBundle(BaseModel):
    """청크 목록과 대응 임베딩 (행 순서 일치)."""

    chunks: List[Chunk]
    vectors: List[List[float]]
    model_name: str
    dim: int


class IndexStats(BaseModel):
    """인덱싱 결과 요약 — 수동 점검용."""

    n_documents: int
    n_articles: int
    n_chunks: int
    dim: int
    per_document: Dict[str, int]
    elapsed_s: float


# =====================================================================================
# 4. 스키마 — 검색 / 증강 / 생성
# =====================================================================================

class RetrievedChunk(BaseModel):
    rank: int
    score: float
    chunk: Chunk


class RetrievalOutput(BaseModel):
    question: str
    hits: List[RetrievedChunk]
    top_k: int
    elapsed_s: float


class PromptBundle(BaseModel):
    """[문서명, 조번호, 본문] 형식으로 조합된 프롬프트."""

    system_prompt: str
    user_prompt: str
    context_block: str
    n_context_chunks: int


class GenerationOutput(BaseModel):
    answer_text: str
    n_new_tokens: int
    elapsed_s: float


class Evidence(BaseModel):
    """retrieved 항목 1건 — [문서명, 조번호] 2원소로 직렬화된다."""

    doc_name: DocName
    article_number: int

    def to_pair(self) -> List[Any]:
        return [self.doc_name, int(self.article_number)]


class AnswerPayload(BaseModel):
    """run_rag_pipeline()의 최종 반환값. 공통 러너 형식 검사와 1:1 대응."""

    answer: str
    retrieved: List[Evidence] = Field(min_length=1, max_length=4)

    @field_validator("retrieved")
    @classmethod
    def _allowed_docs(cls, v: List[Evidence]) -> List[Evidence]:
        for item in v:
            if normalize_doc_name(item.doc_name) not in ALLOWED_DOCS_NORM:
                raise ValueError(f"허용 목록 밖 문서명: {item.doc_name}")
        return v

    def to_contract(self) -> Dict[str, Any]:
        """공통 러너가 기대하는 순수 dict로 변환."""
        return {"answer": self.answer, "retrieved": [e.to_pair() for e in self.retrieved]}


# =====================================================================================
# 5. 스키마 — 품질 결과 (골드셋 / 제출 파일)
# =====================================================================================

class GoldArticle(BaseModel):
    doc: DocName
    article: int
    citation: str


class GoldQuestion(BaseModel):
    id: str
    question: str
    ptype: str
    difficulty: str
    gold_articles: List[GoldArticle]
    key_facts: List[str]


class GoldSet(BaseModel):
    questions: List[GoldQuestion]
    meta: Dict[str, Any] = Field(default_factory=dict, alias="_meta")

    model_config = ConfigDict(populate_by_name=True)


class SubmissionAnswer(BaseModel):
    """answers_public_<팀>.json 의 answers[] 항목."""

    qid: str
    retrieved: List[List[Any]]
    answer: str
    error: Optional[str] = None


class SubmissionFile(BaseModel):
    """answers_public_example.json 과 동일 구조."""

    team: str
    answers: List[SubmissionAnswer]
    meta: Dict[str, Any] = Field(default_factory=dict)


class ArticleScore(BaseModel):
    """근거 조항 정확 일치 채점 결과."""

    qid: str
    predicted: List[List[Any]]
    gold: List[List[Any]]
    hit_at_1: bool
    hit_at_k: bool
    n_gold_matched: int
    n_gold_total: int


class KeyFactScore(BaseModel):
    """정답 핵심 사실 포함 여부. covered는 수동 대조로 확정한다."""

    qid: str
    n_key_facts: int
    n_covered_auto: int
    coverage_auto: float
    per_fact: List[Dict[str, Any]]
    needs_manual_review: bool = True


class ItemReport(BaseModel):
    qid: str
    question: str
    difficulty: str
    ptype: str
    article: ArticleScore
    key_fact: KeyFactScore
    answer_text: str


class EvalReport(BaseModel):
    """공개 10문항 자체 채점 종합."""

    n_items: int
    article_hit_at_1_rate: float
    article_hit_at_k_rate: float
    key_fact_coverage_mean: float
    items: List[ItemReport]


class PerfProtocol(BaseModel):
    """공식 프로토콜과 동일 조건 (Upgrade 단계에서 사용)."""

    requests_per_run: int = 12
    concurrency: int = 2
    warmup_requests: int = 2
    repetitions: int = 3


class PerfReport(BaseModel):
    protocol: PerfProtocol
    success_rate: float
    throughput_rps: float
    p50_latency_s: Optional[float]
    p95_latency_s: Optional[float]


# =====================================================================================
# 6. 인덱싱 — 로딩 및 가져오기
# =====================================================================================

DEFAULT_SOURCES: List[DocumentSource] = [
    DocumentSource(
        doc_name="카카오계정 약관",
        urls=[
            "https://www.kakao.com/policy/terms?lang=ko",
            "https://qr.kakao.com/policy/terms?lang=ko",
            "https://t1.kakaocdn.net/kakaocorp/pw/policy/files/카카오계정약관.pdf",
        ],
        effective_date="2026-05-29",
        note="HTML 실패 시 PDF로 폴백. PDF 경로도 검색 시점 기준 추정값 — 실행 전 재확인 필요.",
    ),
    DocumentSource(
        doc_name="카카오 위치정보 이용약관",
        urls=[
            "https://www.kakao.com/policy/location?lang=ko",
            "https://qr.kakao.com/policy/location?lang=ko",
        ],
        effective_date="2026-07-16",
        note="HTML 경로만 확인됨. PDF 폴백 경로는 미확보.",
    ),
    DocumentSource(
        doc_name="카카오 통합서비스약관",
        urls=[
            "https://www.kakao.com/policy/terms?type=ts&lang=ko",
            "https://qr.kakao.com/policy/terms?type=ts&lang=ko",
        ],
        effective_date="2026-05-29",
        note="URL 실제 접근 가능 여부 확인 필요. PDF 폴백 경로는 미확보.",
    ),
    DocumentSource(
        doc_name="카카오 통합 약관",
        urls=[
            "https://www.kakao.com/policy/kakaoTerms?lang=ko",
        ],
        effective_date="2022-08-25",
        note="반드시 운영진 배포 공식 아카이브 링크(2022-08-25 시행본)로 교체할 것.",
    ),
]


def _load_local_file_text(local_path: str) -> Tuple[str, str]:
    """로컬 PDF/DOCX 경로 → (원문 텍스트, 실제로 읽은 경로).

    Colab에서 노트북과 같은 디렉터리 또는 /content 아래에 파일이 있다고 가정한다.
    """
    from pathlib import Path

    candidates = [Path(local_path), Path("/content") / local_path]
    resolved = next((path for path in candidates if path.exists()), None)
    if resolved is None:
        tried = ", ".join(str(path) for path in candidates)
        raise FileNotFoundError(f"로컬 파일을 찾을 수 없습니다. 시도한 경로: {tried}")

    suffix = resolved.suffix.lower()
    if suffix == ".pdf":
        from pypdf import PdfReader

        reader = PdfReader(str(resolved))
        text = "\n".join(page.extract_text() or "" for page in reader.pages)
    elif suffix == ".docx":
        from docx import Document

        document = Document(str(resolved))
        text = "\n".join(p.text for p in document.paragraphs if p.text.strip())
    else:
        raise ValueError(f"지원하지 않는 로컬 파일 형식: {suffix} ({resolved})")

    return text, str(resolved)


def fetch_document(source: DocumentSource, timeout_s: float = 20.0) -> RawDocument:
    """DocumentSource → RawDocument.

    local_path가 있으면 로컬 PDF/DOCX를 우선 사용한다. 없으면 urls를 순서대로
    시도해 조 구조(제N조)가 확인되는 첫 응답을 채택한다. HTML은 script/style
    제거 후 블록 경계를 개행으로 살려 평문화한다.
    """
    import datetime
    import re

    article_pattern = re.compile(r"제\s*\d+\s*조")

    def finalize(raw_text: str) -> str:
        cleaned = raw_text.replace("\u00a0", " ")
        lines = [line.strip() for line in cleaned.split("\n")]
        return "\n".join(line for line in lines if line)

    if source.local_path:
        raw_text, resolved_path = _load_local_file_text(source.local_path)
        text = finalize(raw_text)
        if len(article_pattern.findall(text)) < 3:
            raise RuntimeError(
                f"[{source.doc_name}] 조 구조 미검출 · 경로={resolved_path} · len={len(text)}"
            )
        return RawDocument(
            doc_name=source.doc_name,
            text=text,
            source_url=resolved_path,
            fetched_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),
            char_len=len(text),
        )

    import requests
    from bs4 import BeautifulSoup

    headers = {
        "User-Agent": (
            "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
            "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
        ),
        "Accept-Language": "ko-KR,ko;q=0.9",
    }
    failures: List[str] = []

    def is_pdf_response(url: str, response: "requests.Response") -> bool:
        content_type = response.headers.get("Content-Type", "").lower()
        return "application/pdf" in content_type or url.lower().endswith(".pdf")

    for url in source.urls:
        try:
            response = requests.get(url, headers=headers, timeout=timeout_s)
            response.raise_for_status()

            if is_pdf_response(url, response):
                from io import BytesIO

                from pypdf import PdfReader

                reader = PdfReader(BytesIO(response.content))
                raw_text = "\n".join(page.extract_text() or "" for page in reader.pages)
            else:
                response.encoding = response.apparent_encoding or "utf-8"
                soup = BeautifulSoup(response.text, "html.parser")
                for tag in soup(["script", "style", "noscript", "header", "footer", "nav"]):
                    tag.decompose()

                # 본문 컨테이너만 스코핑 — wrap_menu(목차)는 이 요소 바깥의
                # 형제 요소라 애초에 섞여 들어오지 않는다. 다른 템플릿의
                # 페이지(예: 카카오 통합 약관)에서 이 선택자가 없으면
                # 전체 페이지로 폴백한다.
                content = soup.select_one("div.wrap_terms.wrap_policy")
                if content is None:
                    print(
                        f"[경고][로딩] {source.doc_name} — "
                        "'div.wrap_terms.wrap_policy' 컨테이너를 찾지 못해 "
                        "전체 페이지에서 추출합니다(목차 혼입 가능, span 휴리스틱에 의존)."
                    )
                    content = soup

                raw_text = content.get_text(separator="\n")

            text = finalize(raw_text)

            if len(article_pattern.findall(text)) < 3:
                failures.append(f"{url}: 조 구조 미검출(len={len(text)})")
                continue

            return RawDocument(
                doc_name=source.doc_name,
                text=text,
                source_url=url,
                fetched_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),
                char_len=len(text),
            )
        except Exception as exc:  # noqa: BLE001 — URL 후보를 끝까지 시도한다
            failures.append(f"{url}: {type(exc).__name__}: {exc}")

    raise RuntimeError(f"[{source.doc_name}] 원문 취득 실패 — " + " | ".join(failures))


def load_documents(
    sources: List[DocumentSource],
    timeout_s: float = 20.0,
) -> List[RawDocument]:
    """List[DocumentSource] → List[RawDocument]."""
    documents: List[RawDocument] = []
    for source in sources:
        document = fetch_document(source, timeout_s)
        print(
            f"[로딩] {document.doc_name} · {document.char_len}자 · {document.source_url}"
        )
        documents.append(document)
    return documents


# =====================================================================================
# 7. 인덱싱 — 파싱
# =====================================================================================

def parse_articles(document: RawDocument) -> List[Article]:
    """RawDocument → List[Article].

    조 번호는 1부터 순차 증가하는 것만 헤더 후보로 인정한다. 다만 목차(TOC)처럼
    "제1조 ... 제N조"가 본문보다 먼저 촘촘하게 나열되는 페이지에서는 이 조건을
    만족하는 1..N 구간이 문서 안에 두 번(목차 1회, 본문 1회) 나타날 수 있다.
    이 경우 각 구간을 독립적으로 모은 뒤, 전체 글자 폭(span)이 가장 큰 구간을
    실제 본문으로 채택한다. 목차는 항목 간 간격이 좁고, 본문은 조마다 실제
    내용이 있어 간격이 훨씬 크다는 점을 이용한다.

    표기 형태 두 가지를 모두 처리한다.
      · "제 1 조 (목적)"  — 공백 있음, 제목 괄호
      · "제1조 목적"      — 공백 없음, 제목 괄호 없음
    """
    import re

    text = document.text
    header_pattern = re.compile(r"제\s*(\d+)\s*조")
    reference_suffix = ("에", "의", "와", "과", "및", "부터", "까지", "에서", ",", "제")

    candidates: List[Tuple[int, int, int, str]] = []  # (시작, 헤더끝, 조번호, 제목)

    for match in header_pattern.finditer(text):
        number = int(match.group(1))

        tail = text[match.end():match.end() + 90]
        stripped_tail = tail.lstrip()
        if stripped_tail and stripped_tail[0] in reference_suffix:
            continue

        title_match = re.match(r"[ \t]*\(([^)\n]{1,60})\)", tail)
        if title_match:
            title = title_match.group(1).strip()
            header_end = match.end() + title_match.end()
        else:
            line_match = re.match(r"[ \t]*([^\n]{0,60})", tail)
            candidate = line_match.group(1).strip() if line_match else ""
            candidate = candidate.split("\n")[0].strip()
            title = candidate if 0 < len(candidate) <= 40 else ""
            header_end = match.end() + (len(line_match.group(0)) if title else 0)

        candidates.append((match.start(), header_end, number, title))

    # 1부터 연속 증가하는 구간(run) 단위로 후보를 분리한다.
    # 목차와 본문이 각각 독립된 1..N 시퀀스를 형성하므로, run이 여러 개 나올 수 있다.
    runs: List[List[Tuple[int, int, int, str]]] = []
    current_run: List[Tuple[int, int, int, str]] = []
    expected_number = 1

    for candidate in candidates:
        _, _, number, _ = candidate
        if number == expected_number:
            current_run.append(candidate)
            expected_number += 1
        elif number == 1:
            if current_run:
                runs.append(current_run)
            current_run = [candidate]
            expected_number = 2
        # 그 외(기대값도 1도 아님)는 잡음으로 보고 무시한다.

    if current_run:
        runs.append(current_run)

    if not runs:
        raise ValueError(f"[{document.doc_name}] 조 구조 파싱 실패 — 정규식 재검토 필요")

    def run_span(run: List[Tuple[int, int, int, str]]) -> int:
        return run[-1][0] - run[0][0]  # 마지막 헤더 시작 - 첫 헤더 시작

    headers = max(runs, key=run_span)

    if len(runs) > 1:
        detail = ", ".join(f"{len(run)}개조/span={run_span(run)}자" for run in runs)
        print(
            f"[경고][파싱] {document.doc_name} 조 시퀀스 {len(runs)}개 발견"
            f"(목차 등 중복 가능성) — {detail} · 가장 긴 구간을 본문으로 채택"
        )

    chapter_pattern = re.compile(r"^제\s*\d+\s*장.*$", re.MULTILINE)
    articles: List[Article] = []

    for index, (_, header_end, number, title) in enumerate(headers):
        body_end = headers[index + 1][0] if index + 1 < len(headers) else len(text)
        body = text[header_end:body_end]
        body = chapter_pattern.sub("", body)
        body = re.sub(r"\n{2,}", "\n", body).strip()

        articles.append(
            Article(
                doc_name=document.doc_name,
                article_number=number,
                article_title=title,
                body=body,
            )
        )

    if not articles:
        raise ValueError(f"[{document.doc_name}] 조 구조 파싱 실패 — 정규식 재검토 필요")

    suspicious = [
        f"제{a.article_number}조(len={len(a.body)})"
        for a in articles
        if len(a.body) < 20
    ]
    if suspicious:
        print(
            f"[경고][파싱] {document.doc_name} 본문이 20자 미만인 조 {len(suspicious)}건 "
            f"— 제목 추출 로직이 본문을 흡수했을 가능성: " + ", ".join(suspicious)
        )

    print(f"[파싱] {document.doc_name} · 제1조~제{articles[-1].article_number}조 "
          f"({len(articles)}개)")
    return articles


# =====================================================================================
# 8. 인덱싱 — 청킹
# =====================================================================================

def chunk_articles(articles: List[Article], config: IndexConfig) -> List[Chunk]:
    """List[Article] → List[Chunk]. 조 단위 1청크, max_chunk_chars 초과분만 분할.

    분할된 조각에도 문서명·조번호 머리말을 붙여 검색 시 조 소속을 유지한다.
    """
    chunks: List[Chunk] = []
    empty_body_articles: List[str] = []

    for article in articles:
        header_line = f"[{article.doc_name}] 제{article.article_number}조"
        if article.article_title:
            header_line += f"({article.article_title})"

        body = article.body
        if not body:
            empty_body_articles.append(f"제{article.article_number}조")
            chunks.append(
                Chunk(
                    chunk_id=(
                        f"{normalize_doc_name(article.doc_name)}"
                        f"-{article.article_number:03d}-00"
                    ),
                    doc_name=article.doc_name,
                    article_number=article.article_number,
                    article_title=article.article_title,
                    text=f"{header_line}\n(본문 파싱 실패 — 원문 확인 필요)",
                )
            )
            continue

        budget = max(200, config.max_chunk_chars - len(header_line) - 1)

        if len(body) <= budget:
            segments = [body]
        else:
            segments = []
            cursor = 0
            while cursor < len(body):
                window_end = min(cursor + budget, len(body))
                if window_end < len(body):
                    boundary = body.rfind("\n", cursor + budget // 2, window_end)
                    if boundary == -1:
                        boundary = body.rfind(". ", cursor + budget // 2, window_end)
                    if boundary != -1:
                        window_end = boundary + 1
                segments.append(body[cursor:window_end].strip())
                cursor = window_end

        segments = [s for s in segments if s] or [""]

        for part_index, segment in enumerate(segments):
            chunks.append(
                Chunk(
                    chunk_id=(
                        f"{normalize_doc_name(article.doc_name)}"
                        f"-{article.article_number:03d}-{part_index:02d}"
                    ),
                    doc_name=article.doc_name,
                    article_number=article.article_number,
                    article_title=article.article_title,
                    text=f"{header_line}\n{segment}",
                )
            )

    if empty_body_articles:
        print(
            f"[경고][청킹] {articles[0].doc_name} 본문 파싱 실패 {len(empty_body_articles)}건: "
            + ", ".join(empty_body_articles)
        )

    print(f"[청킹] {len(articles)}개 조항 → {len(chunks)}개 청크")
    return chunks


# =====================================================================================
# 9. 인덱싱 — 임베딩
# =====================================================================================

class EmbedderHandle(BaseModel):
    """임베딩 모델 핸들. 실제 구현에서는 SentenceTransformer 인스턴스를 보유한다."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    model_name: str
    dim: int
    device: str = "cpu"
    model: Any = None


def build_embedder(config: IndexConfig) -> EmbedderHandle:
    """IndexConfig → EmbedderHandle. bge-m3를 로컬 로드한다."""
    import torch
    from sentence_transformers import SentenceTransformer

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(config.embedding_model_name, device=device)
    dim = int(model.get_sentence_embedding_dimension())

    print(f"[임베딩 모델] {config.embedding_model_name} · device={device} · dim={dim}")
    return EmbedderHandle(
        model_name=config.embedding_model_name, dim=dim, device=device, model=model
    )


def embed_chunks(
    chunks: List[Chunk],
    embedder: EmbedderHandle,
    config: IndexConfig,
) -> EmbeddingBundle:
    """List[Chunk] → EmbeddingBundle. L2 정규화하여 내적=cosine이 되게 한다."""
    texts = [chunk.text for chunk in chunks]
    vectors = embedder.model.encode(
        texts,
        batch_size=config.embedding_batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

    print(f"[임베딩] {len(chunks)}개 청크 · shape={tuple(vectors.shape)}")
    return EmbeddingBundle(
        chunks=chunks,
        vectors=vectors.astype("float32").tolist(),
        model_name=embedder.model_name,
        dim=int(vectors.shape[1]),
    )


# =====================================================================================
# 10. 인덱싱 — 저장
# =====================================================================================

class VectorStoreHandle(BaseModel):
    """FAISS in-memory 저장소 핸들."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    backend: str = "faiss.IndexFlatIP"
    dim: int
    n_vectors: int
    chunks: List[Chunk]
    index: Any = None

    def search(self, query_vector: List[float], top_k: int) -> List[RetrievedChunk]:
        """질의 벡터 → 상위 top_k 청크. 정규화 벡터이므로 내적=cosine."""
        import numpy as np

        query = np.asarray([query_vector], dtype="float32")
        scores, indices = self.index.search(query, min(top_k, self.n_vectors))

        hits: List[RetrievedChunk] = []
        for rank, (score, position) in enumerate(zip(scores[0], indices[0]), start=1):
            if position < 0:
                continue
            hits.append(
                RetrievedChunk(
                    rank=rank,
                    score=round(float(score), 4),
                    chunk=self.chunks[int(position)],
                )
            )
        return hits


def build_vector_store(bundle: EmbeddingBundle) -> VectorStoreHandle:
    """EmbeddingBundle → VectorStoreHandle. FAISS IndexFlatIP 생성 후 add."""
    import faiss
    import numpy as np

    vectors = np.asarray(bundle.vectors, dtype="float32")
    index = faiss.IndexFlatIP(bundle.dim)
    index.add(vectors)

    print(f"[저장] FAISS IndexFlatIP · {index.ntotal}개 벡터 · dim={bundle.dim}")
    return VectorStoreHandle(
        dim=bundle.dim,
        n_vectors=int(index.ntotal),
        chunks=bundle.chunks,
        index=index,
    )


def build_index(config: IndexConfig) -> Tuple[VectorStoreHandle, EmbedderHandle, IndexStats]:
    """인덱싱 오케스트레이션: 로딩 → 파싱 → 청킹 → 임베딩 → 저장."""
    started = time.perf_counter()
    documents = load_documents(config.sources, config.request_timeout_s)

    articles: List[Article] = []
    for document in documents:
        articles.extend(parse_articles(document))

    chunks = chunk_articles(articles, config)
    embedder = build_embedder(config)
    bundle = embed_chunks(chunks, embedder, config)
    store = build_vector_store(bundle)

    per_document: Dict[str, int] = {}
    for chunk in chunks:
        per_document[chunk.doc_name] = per_document.get(chunk.doc_name, 0) + 1

    stats = IndexStats(
        n_documents=len(documents),
        n_articles=len(articles),
        n_chunks=len(chunks),
        dim=store.dim,
        per_document=per_document,
        elapsed_s=round(time.perf_counter() - started, 4),
    )
    return store, embedder, stats


# =====================================================================================
# 11. 검색
# =====================================================================================

def embed_query(
    question: str,
    embedder: EmbedderHandle,
    config: RetrievalConfig,
) -> List[float]:
    """질문 문자열 → 정규화된 질의 벡터."""
    vector = embedder.model.encode(
        [config.query_prefix + question],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )[0]
    return vector.astype("float32").tolist()


def retrieve(
    question: str,
    store: VectorStoreHandle,
    embedder: EmbedderHandle,
    config: RetrievalConfig,
) -> RetrievalOutput:
    """질문 → RetrievalOutput (cosine top-k)."""
    started = time.perf_counter()
    query_vector = embed_query(question, embedder, config)
    hits = store.search(query_vector, config.top_k)
    return RetrievalOutput(
        question=question,
        hits=hits,
        top_k=config.top_k,
        elapsed_s=round(time.perf_counter() - started, 4),
    )


# =====================================================================================
# 12. 증강
# =====================================================================================

SYSTEM_PROMPT = (
    "당신은 카카오 약관 전문 어시스턴트입니다. 제공된 약관 조문만을 근거로 답하십시오. "
    "근거에 없는 내용은 지어내지 말고, 숫자·기간·조문 번호는 근거 그대로 옮기십시오."
)


def build_prompt(retrieval: RetrievalOutput, config: GenerationConfig) -> PromptBundle:
    """RetrievalOutput → PromptBundle. [문서명, 조번호, 본문] 형식으로 조합.

    max_context_chars 예산 안에서 관련도 상위 근거부터 담고, 예산을 넘기는
    근거는 제외한다. 최소 1개는 반드시 포함한다.
    """
    blocks: List[str] = []
    used_chars = 0

    for hit in retrieval.hits:
        block = (
            f"[근거 {hit.rank}] 문서명: {hit.chunk.doc_name} / "
            f"조번호: 제{hit.chunk.article_number}조\n"
            f"본문: {hit.chunk.text}"
        )
        if blocks and used_chars + len(block) > config.max_context_chars:
            break
        if not blocks and len(block) > config.max_context_chars:
            block = block[: config.max_context_chars]
        blocks.append(block)
        used_chars += len(block)

    context_block = "\n\n".join(blocks)
    user_prompt = (
        f"{context_block}\n\n"
        f"질문: {retrieval.question}\n\n"
        "위 근거 조문만을 사용해 한국어로 답하십시오. "
        "답변 안에 근거가 된 조문 번호를 함께 밝히고, "
        "근거에서 확인되지 않는 내용은 '제공된 약관 조문에서 확인할 수 없습니다'라고 답하십시오."
    )

    return PromptBundle(
        system_prompt=SYSTEM_PROMPT,
        user_prompt=user_prompt,
        context_block=context_block,
        n_context_chunks=len(blocks),
    )


# =====================================================================================
# 13. 생성
# =====================================================================================

class GeneratorHandle(BaseModel):
    """Qwen2.5-Instruct 로컬 생성 모델 핸들."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    model_name: str
    load_in_4bit: bool
    model: Any = None
    tokenizer: Any = None


def load_generator(config: GenerationConfig) -> GeneratorHandle:
    """GenerationConfig → GeneratorHandle. T4 기준 4bit 양자화 로드.

    T4(compute capability 7.5)는 bfloat16 가속을 지원하지 않으므로
    연산 dtype은 float16으로 고정한다.
    """
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

    if REQUIRED_GENERATION_MODEL_FAMILY.split("-")[0] not in config.model_name:
        raise ValueError(f"생성 모델은 {REQUIRED_GENERATION_MODEL_FAMILY} 계열이어야 합니다.")

    tokenizer = AutoTokenizer.from_pretrained(config.model_name)

    load_kwargs: Dict[str, Any] = {"device_map": "auto"}
    if config.load_in_4bit and torch.cuda.is_available():
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
        )
    else:
        load_kwargs["torch_dtype"] = torch.float16

    model = AutoModelForCausalLM.from_pretrained(config.model_name, **load_kwargs)
    model.eval()

    print(f"[생성 모델] {config.model_name} · 4bit={config.load_in_4bit}")
    return GeneratorHandle(
        model_name=config.model_name,
        load_in_4bit=config.load_in_4bit,
        model=model,
        tokenizer=tokenizer,
    )


def generate(prompt: PromptBundle, generator: GeneratorHandle) -> GenerationOutput:
    """PromptBundle → GenerationOutput. chat template 적용 후 greedy decoding."""
    import torch

    started = time.perf_counter()
    tokenizer = generator.tokenizer

    messages = [
        {"role": "system", "content": prompt.system_prompt},
        {"role": "user", "content": prompt.user_prompt},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(generator.model.device)

    with torch.inference_mode():
        generated = generator.model.generate(
            **model_inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )

    input_length = model_inputs["input_ids"].shape[1]
    new_token_ids = generated[0][input_length:]
    answer_text = tokenizer.decode(new_token_ids, skip_special_tokens=True).strip()

    return GenerationOutput(
        answer_text=answer_text,
        n_new_tokens=int(new_token_ids.shape[0]),
        elapsed_s=round(time.perf_counter() - started, 4),
    )


def select_evidence(retrieval: RetrievalOutput, max_items: int = 4) -> List[Evidence]:
    """RetrievalOutput → 관련도 순 Evidence 1~4개. (doc, article) 중복 제거."""
    evidence: List[Evidence] = []
    seen: set[Tuple[str, int]] = set()
    for hit in retrieval.hits:
        key = (hit.chunk.doc_name, hit.chunk.article_number)
        if key in seen:
            continue
        seen.add(key)
        evidence.append(
            Evidence(doc_name=hit.chunk.doc_name, article_number=hit.chunk.article_number)
        )
        if len(evidence) >= max_items:
            break
    if not evidence:  # retrieved는 최소 1개여야 형식 검사를 통과한다
        evidence.append(Evidence(doc_name="카카오 통합 약관", article_number=1))
    return evidence


# =====================================================================================
# 14. LangGraph 오케스트레이션
# =====================================================================================

class RagState(BaseModel):
    """LangGraph 상태. 각 노드가 부분 갱신을 반환한다."""

    question: str
    retrieval: Optional[RetrievalOutput] = None
    prompt: Optional[PromptBundle] = None
    generation: Optional[GenerationOutput] = None
    payload: Optional[AnswerPayload] = None


class PipelineContext(BaseModel):
    """그래프 노드가 공유하는 런타임 리소스."""

    model_config = ConfigDict(arbitrary_types_allowed=True)

    store: VectorStoreHandle
    embedder: EmbedderHandle
    generator: GeneratorHandle
    config: PipelineConfig


def make_retrieve_node(ctx: PipelineContext):
    def retrieve_node(state: RagState) -> Dict[str, Any]:
        return {"retrieval": retrieve(state.question, ctx.store, ctx.embedder, ctx.config.retrieval)}

    return retrieve_node


def make_augment_node(ctx: PipelineContext):
    def augment_node(state: RagState) -> Dict[str, Any]:
        assert state.retrieval is not None
        return {"prompt": build_prompt(state.retrieval, ctx.config.generation)}

    return augment_node


def make_generate_node(ctx: PipelineContext):
    def generate_node(state: RagState) -> Dict[str, Any]:
        assert state.prompt is not None
        return {"generation": generate(state.prompt, ctx.generator)}

    return generate_node


def make_finalize_node(ctx: PipelineContext):
    def finalize_node(state: RagState) -> Dict[str, Any]:
        assert state.retrieval is not None and state.generation is not None
        payload = AnswerPayload(
            answer=state.generation.answer_text or "제공된 약관 조문에서 확인할 수 없습니다.",
            retrieved=select_evidence(state.retrieval, ctx.config.retrieval.top_k),
        )
        return {"payload": payload}

    return finalize_node


def build_graph(ctx: PipelineContext):
    """PipelineContext → 컴파일된 LangGraph 그래프."""
    graph = StateGraph(RagState)
    graph.add_node("retrieve", make_retrieve_node(ctx))
    graph.add_node("augment", make_augment_node(ctx))
    graph.add_node("generate", make_generate_node(ctx))
    graph.add_node("finalize", make_finalize_node(ctx))

    graph.set_entry_point("retrieve")
    graph.add_edge("retrieve", "augment")
    graph.add_edge("augment", "generate")
    graph.add_edge("generate", "finalize")
    graph.add_edge("finalize", END)
    return graph.compile()


# =====================================================================================
# 15. 부팅 + 고정 진입점
# =====================================================================================

PIPELINE_CONFIG = PipelineConfig(index=IndexConfig(sources=DEFAULT_SOURCES))

_STORE: Optional[VectorStoreHandle] = None
_EMBEDDER: Optional[EmbedderHandle] = None
_GENERATOR: Optional[GeneratorHandle] = None
_GRAPH: Optional[Any] = None
INDEX_STATS: Optional[IndexStats] = None


def bootstrap(config: PipelineConfig = PIPELINE_CONFIG) -> None:
    """인덱스 + 생성 모델 + 그래프를 준비한다. 셀 실행 시 1회 호출."""
    global _STORE, _EMBEDDER, _GENERATOR, _GRAPH, INDEX_STATS

    if _GRAPH is not None:
        return

    _STORE, _EMBEDDER, INDEX_STATS = build_index(config.index)
    print(f"[인덱싱 완료] {INDEX_STATS.model_dump()}")

    _GENERATOR = load_generator(config.generation)
    ctx = PipelineContext(
        store=_STORE, embedder=_EMBEDDER, generator=_GENERATOR, config=config
    )
    _GRAPH = build_graph(ctx)
    print("[부팅 완료] run_rag_pipeline() 호출 준비됨")


def run_rag_pipeline(question: str) -> Dict[str, Any]:
    """공통 러너 고정 진입점.

    Input : question (비어 있지 않은 str)
    Output: {"answer": str, "retrieved": [[문서명, 조번호], ...]}  (retrieved 1~4개)
    """
    if not isinstance(question, str) or not question.strip():
        raise ValueError("question은 비어 있지 않은 문자열이어야 합니다.")
    if _GRAPH is None:
        raise RuntimeError("bootstrap()이 완료되지 않았습니다.")

    final_state = _GRAPH.invoke(RagState(question=question.strip()))
    payload = final_state["payload"] if isinstance(final_state, dict) else final_state.payload
    if isinstance(payload, dict):
        payload = AnswerPayload(**payload)
    return payload.to_contract()

In [7]:
# -*- coding: utf-8 -*-
# =====================================================================================
#  [학생용] 결과기 개발 기본 틀 — 1번 셀
# =====================================================================================
#  이 셀은 완성된 결과기가 아닙니다. 1번 셀에 팀별 결과기를 구현한 뒤 사용합니다.
#  결과기 코랩은 아래 두 셀을 위에서 아래로 한 번 실행할 수 있어야 합니다.
#
#    1번 셀: 팀별 결과기 구현 — 이 파일의 코드
#    2번 셀: 공개 10문항 공통 러너 — 운영진 배포본, 팀 식별자 한 줄 외 수정 금지
#
#  ┌─ 반드시 유지할 계약 ───────────────────────────────────────────────────────────┐
#  │ · answer_question(question: str) 함수 이름과 입력 형식                         │
#  │ · 반환값: {"answer": 문자열, "retrieved": [[문서명, 조번호], ...]}            │
#  │ · retrieved: 실제 답변에 사용한 근거를 관련도 순으로 1~4개                    │
#  │ · 전역 FastAPI app, GET /health, POST /answer                                 │
#  │ · Qwen2.5-Instruct 계열 생성 모델을 Colab T4에서 로컬 실행                    │
#  │ · 새 Colab T4 런타임에서 외부 준비 작업 없이 위에서 아래로 한 번 실행         │
#  └────────────────────────────────────────────────────────────────────────────────┘
#
#  ┌─ 팀이 자유롭게 구현할 부분 ─────────────────────────────────────────────────────┐
#  │ · 1번 셀 안의 결과기 구현 방식과 필요한 패키지                                 │
#  │ · answer_question 함수 내부의 처리 방식                                        │
#  │   단, 위의 고정 계약과 아래의 금지 조건은 유지해야 합니다.                     │
#  └────────────────────────────────────────────────────────────────────────────────┘
#
#  사용할 수 없는 방식
#    · Google Drive 마운트, 미리 업로드한 파일, 개인 컴퓨터 경로에 의존하는 코드
#    · 외부 생성형 LLM API, 원격 임베딩·리랭커, 원격 관리형 검색 서비스
#    · 실행 중 사람의 파일 업로드·문자 입력·버튼 클릭을 기다리는 코드
#    · torch 재설치, torch.compile
#    · 질문과 관계없이 약관 원문 전체를 매 질문의 프롬프트에 넣는 방식
#
#  주의
#    · 약관 원문을 확보하는 방법은 팀별 자유 구현입니다.
#    · 공개·비공개 답변 JSON은 2번 셀이 생성합니다. 1번 셀에서 직접 만들지 않습니다.
# =====================================================================================


# -------------------------------------------------------------------------------------
# 0. 고정 기준 — 문서명과 생성 모델 계열
# -------------------------------------------------------------------------------------
# retrieved에 기록하는 문서명은 아래 네 이름 중 하나를 그대로 사용합니다.
# 조번호는 3 또는 "제3조"처럼 채점기가 조번호를 식별할 수 있는 형태로 반환합니다.
OFFICIAL_DOCUMENT_NAMES = (
    "카카오계정 약관",
    "카카오 위치정보 이용약관",
    "카카오 통합서비스약관",
    "카카오 통합 약관",
)

# 정확한 모델 크기와 로딩 옵션은 자유지만 생성 모델은 이 계열을 사용합니다.
REQUIRED_GENERATION_MODEL_FAMILY = "Qwen2.5-Instruct"


# =====================================================================================
# 1. 팀별 자유 구현 영역 — 함수 내부를 팀 코드로 교체합니다
# =====================================================================================
# 이 영역에는 팀이 필요한 패키지 설치와 결과기 구현 코드를 작성합니다.
# 구현 방법을 제한하지 않으며, 운영진은 아래 answer_question 함수만 호출합니다.


def answer_question(question: str):
    """공통 러너가 질문마다 호출하는 고정 진입점입니다.

    함수 내부 구현은 자유지만 다음 반환 계약은 반드시 유지합니다.

    return {
        "answer": "질문에 대한 최종 답변",
        "retrieved": [
            ["카카오계정 약관", 3],
            ["카카오 통합서비스약관", 7],
        ],
    }

    retrieved에는 내부 검색 후보 전체가 아니라 실제 답변 생성에 사용한 핵심 근거를
    관련도 순으로 1~4개만 기록합니다.
    """
    # if not isinstance(question, str) or not question.strip():
    #     raise ValueError("question은 비어 있지 않은 문자열이어야 합니다.")
    # raise NotImplementedError(
    #     "answer_question()을 팀별 결과기로 구현하고, "
    #     "answer 문자열과 retrieved 1~4개를 고정 형식으로 반환하세요."
    # )

    bootstrap()  # 최초 1회만 실제 초기화, 이후는 즉시 반환
    return run_rag_pipeline(question)


# =====================================================================================
# 2. 고정 FastAPI 연결 영역 — 삭제하거나 경로를 바꾸지 않습니다
# =====================================================================================
# 2번 공통 러너는 아래 app을 localhost에서 실행하고 다음 주소를 호출합니다.
#   · GET  /health : 결과기 서버 준비 여부 확인
#   · POST /answer : {"question": "..."}을 보내 answer_question() 결과 수신
#
# 팀별 결과기 로직은 위 자유 구현 영역에서 작성합니다. 이 블록은 서버 연결만 담당합니다.
# 동시 요청에서 하나의 GPU 생성 모델이 충돌하지 않도록 Lock을 사용합니다.
import subprocess
import sys
import threading


def _install_server_packages():
    """공통 러너와 연결하는 데 필요한 가벼운 서버 패키지만 설치합니다."""
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "fastapi", "uvicorn"],
        check=True,
    )


_install_server_packages()

from fastapi import FastAPI, HTTPException  # noqa: E402


app = FastAPI(title="KTB AI Performance Result Generator")
_GENERATION_LOCK = threading.Lock()


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/answer")
def answer_api(payload: dict):
    question = payload.get("question")
    if not isinstance(question, str) or not question.strip():
        raise HTTPException(status_code=400, detail="question must be a non-empty string")
    with _GENERATION_LOCK:
        return answer_question(question.strip())


print("[1번 셀 준비] 결과기 구현을 마친 뒤 2번 공통 러너를 실행하세요.")


[1번 셀 준비] 결과기 구현을 마친 뒤 2번 공통 러너를 실행하세요.


In [9]:
# 2번 셀 — 공개 10문항 답변 파일 생성
# 이 셀은 전 팀 공통이며 _SP_TEAM 한 줄 외에는 수정하지 않습니다.
# 새 Google Colab T4 런타임에서 결과기 코드를 먼저 실행한 뒤 이 셀을 실행합니다.
#
# 사용 순서
# 1. 새 Google Colab T4 런타임에서 1번 셀 결과기 코드를 실행합니다.
# 2. 이 공통 러너를 2번 셀에 그대로 둡니다.
# 3. 맨 위 _SP_TEAM에 운영진이 알려준 숫자 팀 식별자를 입력합니다.
# 4. 생성된 answers_public_<팀>.json을 결과기 코랩 파일과 함께 제출합니다.
# 공개 문항 10개 · 실행 방식: http
# ═══════════════════════════════════════════════════════════════
#  ★ 여기 한 줄만 자기 팀으로 바꾸세요. 나머지는 손대지 마세요. ★
# ═══════════════════════════════════════════════════════════════
_SP_TEAM = "3"          # 예: "1"  ← 운영진이 알려준 팀 식별자(숫자)를 그대로 적습니다
# ═══════════════════════════════════════════════════════════════

import builtins as _sp_builtins
import json as _sp_json
import os as _sp_os_rt
import re as _sp_re
import signal as _sp_signal
import socket as _sp_socket
import sys as _sp_sys
import time as _sp_time
import traceback as _sp_traceback
import unicodedata as _sp_unicodedata
import urllib.error as _sp_urlerror
import urllib.request as _sp_urlrequest

_sp_open = _sp_builtins.open
_sp_print = _sp_builtins.print

if "_sp_real_sys_exit" in globals():
    _sp_sys.exit = _sp_real_sys_exit
    if _sp_real_exit is not None:
        _sp_builtins.exit = _sp_real_exit
    if _sp_real_quit is not None:
        _sp_builtins.quit = _sp_real_quit

_SP_OUTPUT_DIR = "/content/"
_SP_OUTPUT_PREFIX = "answers_public_"
_SP_EXPECTED_OUTPUT_PATH = ""
_SP_TEAM_ALLOWED = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz_-"
_SP_TEAM_MAX_LEN = 32
_SP_TEAM_NUMERIC_ONLY = True

def _sp_team_howto(head):
    """중단 사유 + 학생이 바로 고칠 수 있는 안내를 한 덩어리로 만든다."""
    rule = (
        "1 이상의 정수를 문자열로 입력합니다. 예: 1, 2, 17"
        if _SP_TEAM_NUMERIC_ONLY
        else "영문·숫자·밑줄(_)·하이픈(-) 1~" + str(_SP_TEAM_MAX_LEN) + "자"
    )
    return (
        head
        + "\n"
        + "\n  [고치는 법] 이 셀 맨 위 ★ 상자 안의 한 줄을 이렇게 바꾸세요."
        + '\n      _SP_TEAM = "1"      ← 운영진이 알려준 팀 식별자(숫자)를 따옴표 안에 그대로'
        + "\n  [쓸 수 있는 값] " + rule
        + "\n                 띄어쓰기와 / \\ . : 같은 경로 문자는 파일 이름을 깨뜨려 쓸 수 없습니다."
        + "\n  [왜] 결과 파일 이름이 " + _SP_OUTPUT_PREFIX + "<팀>.json 이고, 채점은 이 이름으로"
        + "\n       어느 팀 답안인지 가립니다. 비워 두면 채점 자체가 되지 않습니다."
    )

def _sp_resolve_team(value):
    """_SP_TEAM 을 검사·정리해 돌려준다. 쓸 수 없는 값이면 RuntimeError 로 즉시 중단."""
    if not isinstance(value, str) or not value.strip():
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)가 비어 있어 실행을 중단했습니다. 결과 파일은 만들지 않았습니다."))
    team = value.strip()
    if _SP_TEAM_NUMERIC_ONLY and not _sp_re.fullmatch(r"[1-9][0-9]*", team):
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)는 운영진이 알려준 숫자여야 합니다. 지금 값: " + repr(value)))
    if len(team) > _SP_TEAM_MAX_LEN:
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)가 너무 깁니다(" + str(len(team)) + "자). 팀 이름이 아니라 짧은 식별자입니다."))
    _bad = _sp_builtins.sorted(
        _sp_builtins.set(c for c in team if c not in _SP_TEAM_ALLOWED and not ("가" <= c <= "힣")))
    if _bad:
        raise RuntimeError(_sp_team_howto(
            "★ 팀 식별자(_SP_TEAM)에 파일 이름으로 쓸 수 없는 문자가 있습니다: "
            + ", ".join(repr(c) for c in _bad) + "   (지금 값: " + repr(value) + ")"))
    return team

_SP_TEAM = _sp_resolve_team(_SP_TEAM)
if any(ord(c) > 127 for c in _SP_TEAM):
    _sp_print("[주의] 팀 식별자에 한글 등 ASCII 밖 문자가 있습니다: " + _SP_TEAM
              + " — 운영진이 알려준 식별자가 맞는지 확인하세요."
              " 한글 파일 이름은 내려받기·올리기 과정에서 자모 표현이 달라져 팀이 어긋날 수 있습니다.",
              flush=True)

_SP_OUTPUT_PATH = _SP_OUTPUT_DIR.rstrip("/") + "/" + _SP_OUTPUT_PREFIX + _SP_TEAM + ".json"
if _SP_EXPECTED_OUTPUT_PATH and (_sp_os_rt.path.basename(_SP_OUTPUT_PATH)
                                 != _sp_os_rt.path.basename(_SP_EXPECTED_OUTPUT_PATH)):
    raise RuntimeError(
        "이 셀은 " + _sp_os_rt.path.basename(_SP_EXPECTED_OUTPUT_PATH) + " 용으로 생성됐는데 "
        + _sp_os_rt.path.basename(_SP_OUTPUT_PATH) + " 로 저장하려 합니다"
        "(_SP_TEAM 을 손으로 고쳤습니까?). 다른 팀으로 돌리려면 --team 을 바꿔 셀을 다시 생성하세요."
    )
_SP_AUTO_DOWNLOAD = True
_SP_QUESTIONS_JSON = (
    "[[\"P01\", \"사업자/단체 카카오계정은 계정 정보에 등록된 담당자 몇 명이 이용할 수 있으며, 다른 사람과 공유하는 것은 허용되나요?\"], [\"P02\", \"회사가 예측하거나 통제할 수 없는 사유로 서비스가 중단된 경우, 복구가 몇 시간 이상 지연되면 회사는 공지사항에 게시하여 알리나요?\"], [\"P03\", \"카카오계정 약관에서 회사가 개별 서비스와 연동하여 카카오계정에서 제공한다고 열거한 '카카오계정 서비스'의 내용 5가지는 각각 무엇인가요?\"], [\"P04\", \"회사가 위치기반서비스의 이용을 제한하거나 중지한 때에는 이용자에게 무엇을 어떤 방법으로 알리나요?\"], [\"P05\", \"회사가 위치정보 수집·이용·제공사실 확인자료를 기록·보존하는 근거는 위치정보의 보호 및 이용 등에 관한 법률 제 몇 조 제 몇 항이며, 그 자료는 어디에 기록되어 몇 개월간 보관되나요?\"], [\"P06\", \"카카오계정이 없는 사람이 통합서비스에 가입하려면 무엇을 먼저 해야 하며, 통합서비스 이용계약은 동의·확인·승낙의 어떤 순서로 체결되나요?\"], [\"P07\", \"서비스 명칭에 '카카오'가 사용되더라도 카카오 통합서비스약관의 '통합서비스'에 포함되지 않는 서비스는 누가 제공하는 서비스이며, 약관은 그 예로 무엇을 들고 있나요?\"], [\"P08\", \"카카오 통합 약관과 세부지침(회사가 정한 서비스의 개별 이용약관·운영정책·규칙 등)의 내용이 충돌하는 경우"
    ", 본 약관이 세부지침보다 우선하여 적용되나요?\"], [\"P09\", \"이용자가 서비스 사용을 중단하거나 카카오계정 및 Daum 아이디를 탈퇴한 이후, 게시물에 관하여 회사에 부여한 라이선스의 효력은 어떻게 되나요?\"], [\"P10\", \"8세 이하의 아동 등의 생명 또는 신체 보호를 위해 보호의무자가 개인위치정보의 이용 또는 제공에 동의하려면 어떤 서류에 무엇을 첨부하여 어디에 제출해야 하며, 그 동의는 어떤 효력을 갖나요?\"]]"
)
_SP_QUESTIONS = [tuple(_x) for _x in _sp_json.loads(_SP_QUESTIONS_JSON)]
_SP_ALLOWED_DOCS = _sp_json.loads("[\"카카오계정 약관\", \"카카오 통합서비스약관\", \"카카오 통합 약관\", \"카카오 위치정보 이용약관\"]")
_SP_PER_Q_TIMEOUT_S = 120
_SP_TRANSPORT = "http"
_SP_HTTP_HOST = "127.0.0.1"
_SP_HTTP_PORT = 8765
_SP_HTTP_STARTUP_TIMEOUT_S = 30
_SP_HTTP_HEALTH_PATH = "/health"
_SP_HTTP_ANSWER_PATH = "/answer"
_SP_PERFORMANCE_REQUESTS = 12
_SP_PERFORMANCE_CONCURRENCY = 2
_SP_PERFORMANCE_REPETITIONS = 3
_SP_PERFORMANCE_WARMUP_REQUESTS = 2

_sp_fn = globals().get("answer_question")
if not callable(_sp_fn):
    raise RuntimeError(
        "팀 코드에 answer_question(question) 함수가 없습니다(규정 ②). 실행을 중단합니다."
    )

_sp_doc_warnings = []
_sp_timeouts = []
_sp_http_server = None
_sp_http_thread = None

class _SpHttpTimeout(Exception):
    """HTTP 요청 시간 초과. 품질 추출에서는 timeout_qids로 기록한다."""

def _sp_http_url(path):
    return "http://" + _SP_HTTP_HOST + ":" + str(_SP_HTTP_PORT) + path

def _sp_http_json(method, path, payload=None, timeout_s=None):
    data = None
    headers = {"Accept": "application/json"}
    if payload is not None:
        data = _sp_json.dumps(payload, ensure_ascii=False).encode("utf-8")
        headers["Content-Type"] = "application/json"
    req = _sp_urlrequest.Request(
        _sp_http_url(path), data=data, headers=headers, method=method
    )
    try:
        with _sp_urlrequest.urlopen(req, timeout=timeout_s or _SP_PER_Q_TIMEOUT_S) as resp:
            raw = resp.read().decode("utf-8")
            if resp.status != 200:
                raise RuntimeError("HTTP " + str(resp.status) + ": " + raw[:500])
    except (_sp_socket.timeout, TimeoutError) as exc:
        raise _SpHttpTimeout(str(timeout_s or _SP_PER_Q_TIMEOUT_S) + "초 안에 응답하지 않았습니다.") from exc
    except _sp_urlerror.HTTPError as exc:
        raw = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError("HTTP " + str(exc.code) + ": " + raw[:500]) from exc
    except _sp_urlerror.URLError as exc:
        if isinstance(exc.reason, (_sp_socket.timeout, TimeoutError)):
            raise _SpHttpTimeout(
                str(timeout_s or _SP_PER_Q_TIMEOUT_S) + "초 안에 응답하지 않았습니다."
            ) from exc
        raise RuntimeError("HTTP 연결 실패: " + str(exc.reason)) from exc
    try:
        return _sp_json.loads(raw)
    except _sp_json.JSONDecodeError as exc:
        raise TypeError("HTTP 응답이 JSON이 아닙니다: " + raw[:500]) from exc

def _sp_start_http_server():
    global _sp_http_server, _sp_http_thread
    _sp_app = globals().get("app")
    if _sp_app is None:
        raise RuntimeError(
            "HTTP 실행 모드에는 전역 FastAPI app과 GET /health, POST /answer가 필요합니다."
        )
    try:
        import threading as _sp_threading
        import uvicorn as _sp_uvicorn
    except ImportError as exc:
        raise RuntimeError(
            "HTTP 실행 모드에는 fastapi와 uvicorn이 필요합니다. 팀 설치 목록에 추가하세요."
        ) from exc
    _sp_config = _sp_uvicorn.Config(
        _sp_app,
        host=_SP_HTTP_HOST,
        port=_SP_HTTP_PORT,
        workers=1,
        log_level="warning",
        access_log=False,
    )
    _sp_http_server = _sp_uvicorn.Server(_sp_config)
    _sp_http_thread = _sp_threading.Thread(
        target=_sp_http_server.run, name="ktb-fastapi", daemon=True
    )
    _sp_http_thread.start()
    _sp_deadline = _sp_time.time() + _SP_HTTP_STARTUP_TIMEOUT_S
    _sp_last = None
    while _sp_time.time() < _sp_deadline:
        if not _sp_http_thread.is_alive():
            raise RuntimeError("FastAPI 서버가 준비되기 전에 종료됐습니다.")
        try:
            health = _sp_http_json("GET", _SP_HTTP_HEALTH_PATH, timeout_s=1)
            if isinstance(health, dict):
                _sp_print("[서버] FastAPI /health 준비 완료: " + _sp_http_url(_SP_HTTP_HEALTH_PATH))
                return
        except Exception as exc:
            _sp_last = exc
        _sp_time.sleep(0.2)
    _sp_stop_http_server()
    raise RuntimeError(
        "FastAPI 서버가 " + str(_SP_HTTP_STARTUP_TIMEOUT_S)
        + "초 안에 준비되지 않았습니다: " + str(_sp_last)
    )

def _sp_stop_http_server():
    if _sp_http_server is not None:
        _sp_http_server.should_exit = True
    if _sp_http_thread is not None and _sp_http_thread.is_alive():
        _sp_http_thread.join(timeout=5)

def _sp_invoke(question):
    if _SP_TRANSPORT == "http":
        return _sp_http_json(
            "POST", _SP_HTTP_ANSWER_PATH, {"question": question},
            timeout_s=_SP_PER_Q_TIMEOUT_S,
        )
    return _sp_call_with_timeout(_sp_fn, question, _SP_PER_Q_TIMEOUT_S)

if _SP_TRANSPORT == "http":
    _sp_start_http_server()

_sp_env_warnings = []
for _sp_d in ("/content/drive", "/content/gdrive", "/gdrive"):
    if _sp_os_rt.path.ismount(_sp_d):
        _sp_env_warnings.append(_sp_d + " 가 마운트되어 있습니다")
if _sp_env_warnings:
    _sp_print("", flush=True)
    _sp_print("!" * 86, flush=True)
    _sp_print("[규정 ③ 경고] 이 세션은 운영진 실행 환경과 다릅니다.", flush=True)
    for _sp_w in _sp_env_warnings:
        _sp_print("  · " + _sp_w, flush=True)
    _sp_print("  운영진은 드라이브가 연결되지 않은 새 세션에서 실행합니다. 드라이브에 둔 약관·인덱스를", flush=True)
    _sp_print("  읽고 있다면 본선에서 전량 실패합니다. 약관은 실행 중 내려받거나 셀 안에 포함하세요.", flush=True)
    _sp_print("  확인 방법: 새 노트북을 열어 코드와 이 셀만 붙여 넣고 실행해 보세요.", flush=True)
    _sp_print("!" * 86, flush=True)
    _sp_print("", flush=True)

class _SpTimeout(BaseException):
    """문항 단위 시간 초과.

    **BaseException 을 상속하는 것이 핵심이다.** 팀 코드가 `try/except Exception` 으로
    넓게 감싸는 일은 흔한데, Exception 을 상속하면 그 handler 가 시간 초과를 삼켜
    상한이 무력화된다(그대로 다음 루프를 돌며 계속 매달린다).
    """

def _sp_call_with_timeout(fn, arg, seconds):
    """SIGALRM 으로 문항 호출에 상한을 건다.

    메인 스레드가 아니거나 SIGALRM 이 없는 환경(윈도 등)에서는 signal 설정이
    실패하므로, 그때는 상한 없이 그대로 호출한다 — 상한을 못 걸었다고 해서
    채점 자체를 포기하는 편이 더 나쁘다.

    웹 Colab 셀은 IPython 이 메인 스레드에서 실행하므로 정상 동작한다.
    """
    if not seconds or seconds <= 0:
        return fn(arg)
    _sp_secs = max(1, int(seconds))     # alarm() 은 정수만 받는다. 0 은 '취소' 라 최소 1초.

    def _sp_on_alarm(signum, frame):
        raise _SpTimeout(str(_sp_secs) + "초 안에 응답하지 않았습니다.")

    try:
        _sp_prev = _sp_signal.signal(_sp_signal.SIGALRM, _sp_on_alarm)
        _sp_signal.alarm(_sp_secs)
    except (ValueError, AttributeError, OSError):
        return fn(arg)          # 상한을 걸 수 없는 환경 — 그대로 실행
    try:
        return fn(arg)
    finally:
        _sp_signal.alarm(0)
        try:
            _sp_signal.signal(_sp_signal.SIGALRM, _sp_prev)
        except Exception:
            pass

def _sp_json_safe_art(art):
    """조번호를 JSON 으로 쓸 수 있는 값으로. 표기는 최대한 원본을 살린다.

    **여기서 흡수하지 않으면 30문항을 다 돌린 뒤 파일 저장에서 터진다.**
    일부 수치 라이브러리의 정수형은 dict 도 아니고 2원소 검사도 통과하지만
    json.dump 가 거부한다. 이 값을 흡수하지 않으면
    실패 시점이 맨 끝이라 GPU 시간을 다 쓰고 결과 파일이 없는 최악의 형태가 된다.

    '제7조' 같은 문자열은 그대로 둔다 — 채점기 _art_no 가 정수로 읽는다.
    """
    if isinstance(art, bool):        # bool 은 int 의 하위형이라 먼저 걸러 낸다
        return str(art)
    if isinstance(art, (int, str)):
        return art
    try:                              # np.int64 등 정수로 볼 수 있는 것
        return int(art)
    except (TypeError, ValueError):
        return str(art)

def _sp_norm_doc(x):
    """문서명 대조용 정규화 — NFC 통일 + 공백 전부 제거.

    ⚠️ 채점기 judge_service/engine/objective.py 의 `norm_doc` 과 **같은 규칙이어야 한다.**
    러너는 Colab 셀이라 judge_service 를 import 할 수 없어 규칙을 여기에 복제해 둔다.
    한쪽만 바뀌어 어긋나면 곧바로 오탐이 난다 — 예전에 러너가 완전 일치로 대조하던 때
    '카카오계정약관'·'카카오 계정 약관' 은 실제 채점 MRR 이 1.00 인데도 규정 ④ 위반 경고를
    맞았다. 팀은 없는 문제를 고치러 다니고(자가 확인표가 n_doc_violations == 0 을 요구한다),
    정상 팀이 경고를 맞기 시작하면 아무도 경고를 안 보게 된다.
    두 구현의 일치는 submission_pipeline/tests/test_doc_name_normalization.py 가 고정한다.
    """
    return _sp_re.sub(r"\s+", "", _sp_unicodedata.normalize("NFC", str(x)))

_SP_ALLOWED_DOCS_NORM = _sp_builtins.set(_sp_norm_doc(_d) for _d in _SP_ALLOWED_DOCS)

def _sp_normalize_retrieved(qid, value):
    """retrieved 를 근거순 [[문서명, 조번호], ...] 1~4개로 정규화."""
    if not isinstance(value, (list, tuple)):
        raise TypeError(qid + ": retrieved 는 목록이어야 합니다. (실제: " + type(value).__name__ + ")")
    out = []
    for item in value:
        if isinstance(item, dict) and "doc" in item and "article_no" in item:
            doc, art = item["doc"], item["article_no"]
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            doc, art = item
        else:
            raise TypeError(qid + ": retrieved 항목은 [문서명, 조번호] 2원소여야 합니다. (실제: " + repr(item) + ")")
        doc = str(doc)
        if _SP_ALLOWED_DOCS_NORM and _sp_norm_doc(doc) not in _SP_ALLOWED_DOCS_NORM:
            _sp_doc_warnings.append({"qid": qid, "doc": doc})
        out.append([doc, _sp_json_safe_art(art)])
    if not 1 <= len(out) <= 4:
        raise ValueError(
            qid + ": retrieved 는 실제 답변 근거를 관련도 순으로 1~4개 반환해야 합니다. "
            "(실제: " + str(len(out)) + "개)"
        )
    return out

_sp_answers = []
_sp_errors = []
_sp_total = len(_SP_QUESTIONS)
_sp_print(
    "\n========== " + "공개" + " " + str(_sp_total)
    + "문항 실행 · " + _SP_TEAM + "팀 ==========",
    flush=True,
)
_sp_t0 = _sp_time.time()

for _sp_i, (_sp_qid, _sp_q) in enumerate(_SP_QUESTIONS, 1):
    _sp_print("[" + str(_sp_i).zfill(2) + "/" + str(_sp_total) + "] " + _sp_qid + " 실행 중 ...", flush=True)
    _sp_started = _sp_time.time()
    try:
        _sp_out = _sp_invoke(_sp_q)
        if not isinstance(_sp_out, dict):
            raise TypeError(_sp_qid + ": answer_question() 은 딕셔너리를 반환해야 합니다. (실제: "
                            + type(_sp_out).__name__ + ")")
        _sp_retrieved = _sp_normalize_retrieved(_sp_qid, _sp_out.get("retrieved"))
        _sp_answer = _sp_out.get("answer")
        if not isinstance(_sp_answer, str):
            raise TypeError(_sp_qid + ": answer 는 문자열이어야 합니다. (실제: "
                            + type(_sp_answer).__name__ + ")")
        _sp_answers.append({"qid": _sp_qid, "retrieved": _sp_retrieved, "answer": _sp_answer})
    except (_SpTimeout, _SpHttpTimeout) as _sp_exc:  # 한 문항이 세션 전체를 잡아먹지 않도록 끊는다.
        _sp_msg = "Timeout: " + str(_sp_exc)
        _sp_timeouts.append(_sp_qid)
        _sp_errors.append({"qid": _sp_qid, "error": _sp_msg})
        _sp_answers.append({"qid": _sp_qid, "retrieved": [], "answer": "", "error": _sp_msg})
        _sp_print("[시간초과] " + _sp_qid + " — " + _sp_msg, flush=True)
    except Exception as _sp_exc:  # 한 문항 실패로 30문항 전체를 잃지 않는다.
        _sp_msg = type(_sp_exc).__name__ + ": " + str(_sp_exc)
        _sp_errors.append({"qid": _sp_qid, "error": _sp_msg})
        _sp_answers.append({"qid": _sp_qid, "retrieved": [], "answer": "", "error": _sp_msg})
        _sp_print("[오류] " + _sp_qid + " — " + _sp_msg, flush=True)
        _sp_traceback.print_exc()
    finally:
        _sp_print("      (" + str(round(_sp_time.time() - _sp_started, 1)) + "s)", flush=True)

_sp_performance = None
if _SP_TRANSPORT == "http" and _SP_PERFORMANCE_REQUESTS > 0:
    from concurrent.futures import ThreadPoolExecutor as _SpThreadPoolExecutor

    def _sp_perf_one(index):
        _qid, _question = _SP_QUESTIONS[index % len(_SP_QUESTIONS)]
        started = _sp_time.perf_counter()
        try:
            value = _sp_http_json(
                "POST", _SP_HTTP_ANSWER_PATH, {"question": _question},
                timeout_s=_SP_PER_Q_TIMEOUT_S,
            )
            ok = (
                isinstance(value, dict)
                and isinstance(value.get("answer"), str)
                and isinstance(value.get("retrieved"), (list, tuple))
            )
            return {
                "ok": ok,
                "qid": _qid,
                "latency_s": round(_sp_time.perf_counter() - started, 4),
                "error": None if ok else "invalid_schema",
            }
        except Exception as exc:
            return {
                "ok": False,
                "qid": _qid,
                "latency_s": round(_sp_time.perf_counter() - started, 4),
                "error": type(exc).__name__ + ": " + str(exc),
            }

    def _sp_percentile(values, ratio):
        if not values:
            return None
        pos = min(len(values) - 1, max(0, int((len(values) - 1) * ratio)))
        return round(values[pos], 4)

    def _sp_median(values):
        values = sorted(values)
        if not values:
            return None
        middle = len(values) // 2
        if len(values) % 2:
            return values[middle]
        return (values[middle - 1] + values[middle]) / 2

    def _sp_perf_round(n_requests, repetition):
        started = _sp_time.perf_counter()
        with _SpThreadPoolExecutor(max_workers=max(1, _SP_PERFORMANCE_CONCURRENCY)) as pool:
            rows = list(pool.map(_sp_perf_one, range(n_requests)))
        wall_s = _sp_time.perf_counter() - started
        ok_rows = [row for row in rows if row["ok"]]
        latencies = sorted(row["latency_s"] for row in ok_rows)
        return {
            "repetition": repetition,
            "transport": "http",
            "requests": n_requests,
            "concurrency": _SP_PERFORMANCE_CONCURRENCY,
            "success": len(ok_rows),
            "fail": len(rows) - len(ok_rows),
            "success_rate": round(len(ok_rows) / len(rows), 4),
            "throughput_rps": round(len(ok_rows) / wall_s, 4) if wall_s else 0.0,
            "wall_s": round(wall_s, 4),
            "p50_latency_s": _sp_percentile(latencies, 0.50),
            "p95_latency_s": _sp_percentile(latencies, 0.95),
            "errors": [row for row in rows if not row["ok"]],
        }

    _sp_warmup = None
    if _SP_PERFORMANCE_WARMUP_REQUESTS > 0:
        _sp_print(
            "[성능] 워밍업 " + str(_SP_PERFORMANCE_WARMUP_REQUESTS) + "요청 실행 중 ...",
            flush=True,
        )
        _sp_warmup = _sp_perf_round(_SP_PERFORMANCE_WARMUP_REQUESTS, 0)

    _sp_perf_samples = []
    for _sp_repetition in range(1, _SP_PERFORMANCE_REPETITIONS + 1):
        _sp_print(
            "[성능] 측정 " + str(_sp_repetition) + "/"
            + str(_SP_PERFORMANCE_REPETITIONS) + " 실행 중 ...",
            flush=True,
        )
        _sp_perf_samples.append(
            _sp_perf_round(_SP_PERFORMANCE_REQUESTS, _sp_repetition)
        )

    _sp_success_median = _sp_median([row["success"] for row in _sp_perf_samples])
    _sp_fail_median = _sp_median([row["fail"] for row in _sp_perf_samples])
    _sp_p50_values = [
        row["p50_latency_s"] for row in _sp_perf_samples
        if row["p50_latency_s"] is not None
    ]
    _sp_p95_values = [
        row["p95_latency_s"] for row in _sp_perf_samples
        if row["p95_latency_s"] is not None
    ]
    _sp_performance = {
        "version": 2,
        "transport": "http",
        "requests": _SP_PERFORMANCE_REQUESTS,
        "concurrency": _SP_PERFORMANCE_CONCURRENCY,
        "success": int(_sp_success_median),
        "fail": int(_sp_fail_median),
        "success_rate": round(_sp_median(
            [row["success_rate"] for row in _sp_perf_samples]
        ), 4),
        "throughput_rps": round(_sp_median(
            [row["throughput_rps"] for row in _sp_perf_samples]
        ), 4),
        "wall_s": round(_sp_median(
            [row["wall_s"] for row in _sp_perf_samples]
        ), 4),
        "p50_latency_s": (
            round(_sp_median(_sp_p50_values), 4) if _sp_p50_values else None
        ),
        "p95_latency_s": (
            round(_sp_median(_sp_p95_values), 4) if _sp_p95_values else None
        ),
        "errors": [
            dict(error, repetition=sample["repetition"])
            for sample in _sp_perf_samples
            for error in sample["errors"]
        ],
        "summary_method": "median",
        "protocol": {
            "requests_per_run": _SP_PERFORMANCE_REQUESTS,
            "concurrency": _SP_PERFORMANCE_CONCURRENCY,
            "warmup_requests": _SP_PERFORMANCE_WARMUP_REQUESTS,
            "repetitions": _SP_PERFORMANCE_REPETITIONS,
        },
        "samples": _sp_perf_samples,
    }
    if _sp_warmup is not None:
        _sp_performance["warmup"] = _sp_warmup
    _sp_print(
        "[성능] closed-loop 중앙값 · "
        + str(_SP_PERFORMANCE_REQUESTS) + "요청 × "
        + str(_SP_PERFORMANCE_REPETITIONS) + "회 · 동시성 "
        + str(_SP_PERFORMANCE_CONCURRENCY) + " · 대표 성공 "
        + str(_sp_performance["success"]) + " · "
        + str(_sp_performance["throughput_rps"]) + " req/s · p95 "
        + str(_sp_performance["p95_latency_s"]) + "s",
        flush=True,
    )

_sp_stop_http_server()

_sp_submission = {"team": _SP_TEAM, "answers": _sp_answers}
if _sp_doc_warnings or _sp_timeouts or _sp_env_warnings or _sp_performance:
    _sp_submission["meta"] = {"doc_name_violations": _sp_doc_warnings,
                              "timeout_qids": _sp_timeouts,
                              "env_warnings": _sp_env_warnings,
                              "transport": _SP_TRANSPORT}
    if _sp_performance:
        _sp_submission["meta"]["performance"] = _sp_performance
_sp_text = _sp_json.dumps(_sp_submission, ensure_ascii=False, indent=2, default=str)
with _sp_open(_SP_OUTPUT_PATH, "w", encoding="utf-8") as _sp_f:
    _sp_f.write(_sp_text)

_sp_print("[완료] " + str(len(_sp_answers)) + "문항 저장: " + _SP_OUTPUT_PATH
      + "  (총 " + str(round(_sp_time.time() - _sp_t0, 1)) + "s)", flush=True)
if _sp_errors:
    _sp_print("[경고] 실패 문항 " + str(len(_sp_errors)) + "건: "
          + ", ".join(_e["qid"] for _e in _sp_errors), flush=True)
if _sp_doc_warnings:
    _sp_print("[경고] 규정 ④ 위반 — 허용 목록 밖 문서명 " + str(len(_sp_doc_warnings)) + "건: "
          + ", ".join(sorted(set(_w["doc"] for _w in _sp_doc_warnings)))
          + "  → 해당 항목은 검색 점수가 0으로 채점됩니다. 허용(띄어쓰기 차이는 무관): "
          + ", ".join(_SP_ALLOWED_DOCS), flush=True)

if _SP_AUTO_DOWNLOAD:
    try:
        from google.colab import files as _sp_files
        _sp_files.download(_SP_OUTPUT_PATH)
        _sp_print("[다운로드] 브라우저 다운로드를 시작했습니다: " + _SP_OUTPUT_PATH, flush=True)
    except Exception as _sp_dl_exc:
        _sp_print("[다운로드] 자동 다운로드 실패(" + type(_sp_dl_exc).__name__ + ": " + str(_sp_dl_exc)
                  + ") — 좌측 파일 탭에서 " + _SP_OUTPUT_PATH + " 를 직접 내려받으세요.", flush=True)

_sp_print("SUBMISSION_RUNNER_DONE " + _sp_json.dumps(
    {"team": _SP_TEAM, "output_path": _SP_OUTPUT_PATH, "n_answers": len(_sp_answers),
     "n_errors": len(_sp_errors), "failed_qids": [_e["qid"] for _e in _sp_errors],
     "n_doc_violations": len(_sp_doc_warnings), "timeout_qids": _sp_timeouts,
     "env_warnings": _sp_env_warnings, "transport": _SP_TRANSPORT,
     "performance": _sp_performance},
    ensure_ascii=False), flush=True)


[서버] FastAPI /health 준비 완료: http://127.0.0.1:8765/health

========== 공개 10문항 실행 · 3팀 ==========
[01/10] P01 실행 중 ...
[로딩] 카카오계정 약관 · 11162자 · https://www.kakao.com/policy/terms?lang=ko
[경고][로딩] 카카오 위치정보 이용약관 — 'div.wrap_terms.wrap_policy' 컨테이너를 찾지 못해 전체 페이지에서 추출합니다(목차 혼입 가능, span 휴리스틱에 의존).
[로딩] 카카오 위치정보 이용약관 · 5951자 · https://www.kakao.com/policy/location?lang=ko
[로딩] 카카오 통합서비스약관 · 15059자 · https://www.kakao.com/policy/terms?type=ts&lang=ko
[로딩] 카카오 통합 약관 · 18005자 · https://www.kakao.com/policy/kakaoTerms?lang=ko
[파싱] 카카오계정 약관 · 제1조~제17조 (17개)
[파싱] 카카오 위치정보 이용약관 · 제1조~제16조 (16개)
[파싱] 카카오 통합서비스약관 · 제1조~제18조 (18개)
[파싱] 카카오 통합 약관 · 제1조~제21조 (21개)
[청킹] 72개 조항 → 77개 청크


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

[생성 모델] Qwen/Qwen2.5-7B-Instruct · 4bit=True
[부팅 완료] run_rag_pipeline() 호출 준비됨
      (75.0s)
[05/10] P05 실행 중 ...
      (8.7s)
[06/10] P06 실행 중 ...
      (10.8s)
[07/10] P07 실행 중 ...
      (7.8s)
[08/10] P08 실행 중 ...
      (12.4s)
[09/10] P09 실행 중 ...
      (12.2s)
[10/10] P10 실행 중 ...
      (9.0s)
[성능] 워밍업 2요청 실행 중 ...
[성능] 측정 1/3 실행 중 ...
[성능] 측정 2/3 실행 중 ...
[성능] 측정 3/3 실행 중 ...
[성능] closed-loop 중앙값 · 12요청 × 3회 · 동시성 2 · 대표 성공 12 · 0.0939 req/s · p95 38.482s
[완료] 10문항 저장: /content/answers_public_3.json  (총 894.0s)
[경고] 실패 문항 3건: P01, P02, P03


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[다운로드] 브라우저 다운로드를 시작했습니다: /content/answers_public_3.json
SUBMISSION_RUNNER_DONE {"team": "3", "output_path": "/content/answers_public_3.json", "n_answers": 10, "n_errors": 3, "failed_qids": ["P01", "P02", "P03"], "n_doc_violations": 0, "timeout_qids": ["P01", "P02", "P03"], "env_warnings": [], "transport": "http", "performance": {"version": 2, "transport": "http", "requests": 12, "concurrency": 2, "success": 12, "fail": 0, "success_rate": 1.0, "throughput_rps": 0.0939, "wall_s": 127.8311, "p50_latency_s": 18.6998, "p95_latency_s": 38.482, "errors": [], "summary_method": "median", "protocol": {"requests_per_run": 12, "concurrency": 2, "warmup_requests": 2, "repetitions": 3}, "samples": [{"repetition": 1, "transport": "http", "requests": 12, "concurrency": 2, "success": 12, "fail": 0, "success_rate": 1.0, "throughput_rps": 0.0937, "wall_s": 128.0646, "p50_latency_s": 18.6636, "p95_latency_s": 38.6645, "errors": []}, {"repetition": 2, "transport": "http", "requests": 12, "concurrency": 2,

In [12]:
import base64
from IPython.display import HTML

with open('/content/answers_public_3.json', 'rb') as f:
    data = f.read()

b64 = base64.b64encode(data).decode()
HTML(f'<a download="result.json" href="data:application/json;base64,{b64}">json 다운로드</a>')